# Predictive Maintenance & Anomaly Detection
### Industrial Machine Health Monitoring with AI4I 2020
**Role / Persona:** Data & AI Consultant / Senior Machine Learning Engineer  
**Objective:** End-to-end predictive failure modeling, unsupervised sensor outlier detection, domain feature engineering, and operational risk assessment.

---

## Executive Summary & Business Architecture
Industrial milling machines operate under rigorous thermo-mechanical stress. Unplanned downtime leads to catastrophic tool fractures, ruined workpieces, factory line stoppages, and expensive emergency technician dispatches. 

This project implements a **dual-layer machine learning architecture**:
1. **Supervised Failure Prediction:** Predicts the probability of imminent machine failure ($y \in \{0, 1\}$) conditioned on operational telemetry.
2. **Unsupervised Anomaly Detection:** Employs Isolation Forest to identify statistical operating outliers in multi-dimensional space without requiring target labels.


### 1. Environment Setup & Dependency Imports

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Add src package to sys.path
sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_data, audit_data_quality
from src.feature_engineering import MachineFeatureEngineer, engineer_features
from src.preprocessing import split_data, fit_and_save_preprocessors
from src.evaluation import compute_metrics
from src.prediction import MaintenancePredictor

print("Environment configured successfully!")


### 2. Data Loading & Data Quality Audit
We load the raw dataset (`data/ai4i2020.csv`) and conduct a comprehensive quality and physical plausibility audit.


In [ ]:
# Load validated dataset
df = load_data('../data/ai4i2020.csv')
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()


In [ ]:
# Comprehensive Quality Audit
audit_results = audit_data_quality(df)
print("Missing values total:", audit_results['missing_values']['total_missing'])
print("Duplicate rows:", audit_results['duplicate_records']['exact_duplicate_rows'])
print("Product Type Distribution:", audit_results['type_distribution']['percentages'])
print("Target Distribution:", audit_results['target_distribution']['percentages'])
print("Class Imbalance Ratio:", audit_results['target_distribution']['imbalance_ratio'])
print("Physics Checks Status:", "PASSED" if audit_results['plausibility_and_physics_checks']['all_physics_checks_passed'] else "FAILED")


### 3. Exploratory Data Analysis (EDA)
Analyzing target class imbalance, continuous parameter distributions, median shifts during failure, and physical failure mechanisms.


In [ ]:
# Target Class Imbalance & Quality Variant Failure Rate
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

target_counts = df['Machine failure'].value_counts()
axes[0].bar(['Normal (0)', 'Failure (1)'], target_counts.values, color=['#2b5c8f', '#d95f02'], edgecolor='black', width=0.5)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 120, f"{v:,}\n({v/len(df)*100:.2f}%)", ha='center', fontweight='bold')
axes[0].set_title("Machine Failure Distribution (Imbalance Ratio 28.5:1)")
axes[0].set_ylabel("Count")

type_stats = df.groupby('Type')['Machine failure'].agg(['count', 'mean'])
type_stats['mean_pct'] = type_stats['mean'] * 100
axes[1].bar(type_stats.index, type_stats['mean_pct'], color=['#4575b4', '#74add1', '#abd9e9'], edgecolor='black', width=0.5)
for idx, val in enumerate(type_stats['mean_pct']):
    axes[1].text(idx, val + 0.1, f"{val:.2f}%", ha='center', fontweight='bold')
axes[1].set_title("Failure Rate by Product Quality Variant (L vs M vs H)")
axes[1].set_ylabel("Failure Rate (%)")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation Matrix
num_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='Blues', vmin=-1, vmax=1, center=0, square=True)
plt.title("Pearson Correlation Heatmap (Continuous Telemetry & Failure)")
plt.show()


### 4. Domain-Driven Physical Feature Engineering
Derived from mechanical and thermodynamic first principles:
- **`Temp_Difference_K`**: Heat dissipation gradient ($T_{\text{process}} - T_{\text{air}}$).
- **`Temp_Ratio`**: Dimensionless thermal expansion ratio ($T_{\text{process}} / T_{\text{air}}$).
- **`Power_W`**: Spindle mechanical output power ($\tau \cdot \omega = \tau \cdot \text{rpm} \cdot \frac{2\pi}{60}$).
- **`Overstrain_Torque_Wear`**: Stress product ($\tau \cdot \text{Tool wear}$).


In [ ]:
# Execute Feature Engineering
df_eng = engineer_features(df, drop_leakage_cols=True)
print("Engineered Features Added:", ['Temp_Difference_K', 'Temp_Ratio', 'Power_W', 'Overstrain_Torque_Wear'])
df_eng[['Temp_Difference_K', 'Temp_Ratio', 'Power_W', 'Overstrain_Torque_Wear']].describe().T[['mean', 'std', 'min', '50%', 'max']]


### 5. Stratified Train / Test Split
To avoid sampling variance under 28.5:1 class imbalance, we enforce an identical 3.39% failure incidence in both train and test partitions.


In [ ]:
X_train, X_test, y_train, y_test, split_info = split_data(df, random_state=42)
print("Training Set:", split_info['training_set'])
print("Test Set:", split_info['test_set'])


### 6. Supervised Model Training & Evaluation
We evaluate Logistic Regression (Baseline), Random Forest, and XGBoost on the holdout test set ($2,000$ samples, $68$ failures).


In [ ]:
# Load pre-computed hold-out comparison matrix
comp_df = pd.read_csv('../outputs/metrics/model_comparison.csv')
comp_df


In [ ]:
# Display Evaluation Figures
from IPython.display import Image, display

print("--- Side-by-Side Confusion Matrices ---")
display(Image(filename='../outputs/figures/06_confusion_matrices.png'))

print("--- ROC and Precision-Recall Curves ---")
display(Image(filename='../outputs/figures/07_roc_pr_curves.png'))

print("--- Feature Importance Ranking ---")
display(Image(filename='../outputs/figures/08_feature_importances.png'))

print("--- SHAP TreeExplainer Global Interpretability ---")
display(Image(filename='../outputs/figures/09_shap_summary.png'))


### 7. Unsupervised Anomaly Detection (Isolation Forest)
Trained strictly on multi-dimensional continuous operating features **without target labels**.


In [ ]:
with open('../outputs/metrics/anomaly_detection_summary.json', 'r') as f:
    anom_summary = json.load(f)

print(f"Total Observations: {anom_summary['detection_results']['total_observations']:,}")
print(f"Detected Anomalies: {anom_summary['detection_results']['anomalous_observations']} ({anom_summary['detection_results']['anomaly_rate_percentage']}%)")
print("Exploratory Failure Overlap:")
print(f"  Failures in Anomaly Space: {anom_summary['exploratory_overlap_with_machine_failure']['actual_failures_flagged_as_anomalies']} of 339 ({anom_summary['exploratory_overlap_with_machine_failure']['exploratory_failure_capture_rate_pct']}%)")
display(Image(filename='../outputs/figures/10_anomaly_score_distribution.png'))
display(Image(filename='../outputs/figures/11_anomaly_scatter_profiles.png'))


### 8. Production Inference Demonstration
Demonstrating real-time prediction using the serialized pipeline.


In [ ]:
predictor = MaintenancePredictor(
    failure_pipeline_path='../models/failure_prediction_pipeline.pkl',
    anomaly_model_path='../models/isolation_forest.pkl'
)

# Representative test scenario: High Wear + High Torque
test_input = {
    'Type': 'L',
    'Air temperature [K]': 302.5,
    'Process temperature [K]': 311.2,
    'Rotational speed [rpm]': 1280,
    'Torque [Nm]': 65.5,
    'Tool wear [min]': 210,
}

result = predictor.predict_machine_state(test_input)
for k, v in result.items():
    print(f"{k}: {v}")


### 9. Key Technical & Interview Defense Points
1. **Why Accuracy is Deceptive:** Under 3.39% class imbalance, predicting all zeros yields 96.61% accuracy while catching zero catastrophic failures. F1-Score and PR-AUC are the true north star metrics.
2. **Target Leakage Strictness:** The failure mode decomposition columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) and identifier columns (`UDI`, `Product ID`) were strictly quarantined from the training set.
3. **Domain Feature Engineering Supremacy:** The engineered feature `Overstrain_Torque_Wear` ranked #1 in global feature importance across both Random Forest and XGBoost, outperforming all raw sensor variables.
4. **Supervised vs. Unsupervised Synergy:** Supervised classification catches known historical breakdown patterns with high precision (94.74%), while unsupervised anomaly detection serves as an early-warning telemetry guardrail against unprecedented operational regimes.
